In [2]:
import sys
sys.path.append('/home/cloud/Desktop/abhi/VillainNet/')

import torch
import torch.nn as nn
from itertools import permutations
import copy

from CompOFA.ofa.elastic_nn.networks.ofa_mbv3 import OFAMobileNetV3
from CompOFA.ofa.elastic_nn.networks.ofa_resnet import OFAResNets
from CompOFA.ofa.imagenet_codebase.utils.pytorch_utils import get_net_info
import numpy as np


/home/david/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# GTSRB
mobilenet = OFAMobileNetV3(n_classes=43, bn_param=(0.1, 1e-5), base_stage_width='proxyless', width_mult_list=[1.0],
                             dropout_rate=0.1, ks_list=[3, 5, 7], expand_ratio_list=[3, 4, 6], depth_list=[2, 3, 4],
                             compound=False, fixed_kernel=True)

resnet = OFAResNets(n_classes=43, bn_param=(0.1, 1e-5), width_mult_list=[1.0],
                             dropout_rate=0.1, expand_ratio_list=[3, 4, 6], depth_list=[2, 3, 4],
                             compound=False, fixed_kernel=True)

# CIFAR10
mobilenet_cifar = OFAMobileNetV3(n_classes=10, bn_param=(0.1, 1e-5), base_stage_width='proxyless', width_mult_list=[1.0],
                             dropout_rate=0.1, ks_list=[3, 5, 7], expand_ratio_list=[3, 4, 6], depth_list=[2, 3, 4],
                             compound=False, fixed_kernel=True)

resnet_cifar = OFAResNets(n_classes=10, bn_param=(0.1, 1e-5), width_mult_list=[1.0],
                             dropout_rate=0.1, expand_ratio_list=[3, 4, 6], depth_list=[2, 3, 4],
                             compound=False, fixed_kernel=True)

In [17]:
mobilenet_gtsrb_flops = {}
resnet_gtsrb_flops = {}
mobilenet_cifar_flops = {}
resnet_cifar_flops = {}

# Given values
depth_values = [2, 3, 4]  # Possible depth values
mapping = {2: 3, 3: 4, 4: 6}  # Depth-to-expand ratio mapping

# Generate all unique permutations of depth_list with length 5
depth_permutations = set(permutations(depth_values * 5, 5))

# # Generate corresponding expand_ratio_list for each permutation
all_combinations = []
for depth_list in depth_permutations:
    expand_ratio_list = [mapping[d] for d in depth_list for _ in range(4)]
    all_combinations.append((depth_list, expand_ratio_list))

print(f"Total combinations: {len(all_combinations)}")

for combination in all_combinations:
    depth_list, expand_ratio_list = combination
    mobilenet.set_active_subnet(None, None, expand_ratio_list, depth_list)
    resnet.set_active_subnet(None, None, expand_ratio_list, depth_list)
    mobilenet_cifar.set_active_subnet(None, None, expand_ratio_list, depth_list)
    resnet_cifar.set_active_subnet(None, None, expand_ratio_list, depth_list)

    # Calculate FLOPs for MobileNetV3
    sub = mobilenet.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")
    flops_mobilenet = subnet_info['flops'] / 1e6
    if flops_mobilenet in mobilenet_gtsrb_flops:
        # If this flops value already exists, append the combination
        mobilenet_gtsrb_flops[flops_mobilenet].append(combination)
    else:
        # If this flops value does not exist, create a new entry
        mobilenet_gtsrb_flops[flops_mobilenet] = [combination]

    # Calculate FLOPs for ResNet
    sub = resnet.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")
    flops_resnet = subnet_info['flops'] / 1e6
    if flops_resnet in resnet_gtsrb_flops:
        # If this flops value already exists, append the combination
        resnet_gtsrb_flops[flops_resnet].append(combination)
    else:
        # If this flops value does not exist, create a new entry
        resnet_gtsrb_flops[flops_resnet] = [combination]

    # Calculate FLOPs for MobileNetV3 CIFAR
    sub = mobilenet_cifar.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")
    flops_mobilenet_cifar = subnet_info['flops'] / 1e6
    if flops_mobilenet_cifar in mobilenet_cifar_flops:
        # If this flops value already exists, append the combination
        mobilenet_cifar_flops[flops_mobilenet_cifar].append(combination)
    else:
        # If this flops value does not exist, create a new entry
        mobilenet_cifar_flops[flops_mobilenet_cifar] = [combination]
    
    # Calculate FLOPs for ResNet CIFAR
    sub = resnet_cifar.get_active_subnet(preserve_weight=True)
    subnet_info = get_net_info(sub, measure_latency="gpu16")
    flops_resnet_cifar = subnet_info['flops'] / 1e6
    if flops_resnet_cifar in resnet_cifar_flops:
        # If this flops value already exists, append the combination
        resnet_cifar_flops[flops_resnet_cifar].append(combination)


Total combinations: 243
Total training params: 4.08M
Total FLOPs: 242.52M
Estimated gpu16 latency: 63.729ms
Total training params: 463.26M
Total FLOPs: 51012.15M
Estimated gpu16 latency: 1971.404ms
Total training params: 4.04M
Total FLOPs: 242.47M
Estimated gpu16 latency: 63.446ms
Total training params: 463.24M
Total FLOPs: 51012.14M
Estimated gpu16 latency: 1949.628ms
Total training params: 2.78M
Total FLOPs: 217.11M
Estimated gpu16 latency: 60.582ms
Total training params: 296.11M
Total FLOPs: 47788.04M
Estimated gpu16 latency: 1739.380ms
Total training params: 2.74M
Total FLOPs: 217.07M
Estimated gpu16 latency: 61.212ms
Total training params: 296.10M
Total FLOPs: 47788.03M
Estimated gpu16 latency: 1732.087ms
Total training params: 5.08M
Total FLOPs: 341.94M
Estimated gpu16 latency: 169.124ms
Total training params: 611.85M
Total FLOPs: 75366.38M
Estimated gpu16 latency: 2815.347ms
Total training params: 5.04M
Total FLOPs: 341.90M
Estimated gpu16 latency: 180.995ms
Total training param

In [20]:
np.save('lookup_tables/mobilenet_gtsrb_flops.npy', mobilenet_gtsrb_flops)
np.save('lookup_tables/resnet_gtsrb_flops.npy', resnet_gtsrb_flops)
np.save('lookup_tables/mobilenet_cifar_flops.npy', mobilenet_cifar_flops)
np.save('lookup_tables/resnet_cifar_flops.npy', resnet_cifar_flops)

In [3]:
mobile_net_gtsrb_flops = np.load('lookup_tables/mobilenet_gtsrb_flops.npy', allow_pickle=True).item()

In [11]:
for key in mobile_net_gtsrb_flops.keys():
    if (len(mobile_net_gtsrb_flops[key]) > 1):
        print(f"Key: {key}, Values: {mobile_net_gtsrb_flops[key]}")

In [4]:
import collections
od = collections.OrderedDict(sorted(mobile_net_gtsrb_flops.items()))

In [4]:
def get_arch_edit_distance(target_subnet, random_subnet):
    '''

        Input to this function are the subnet settings for the target and random subnets
        e.g.
            [[3, 3, 3, 3, 3, 3, 3,3, ,3 ,3, ], [2, 2, 2, 2, 2]]
            target_subnet: [[[3, 3, 3, 3], 2], [[4, 4, 4, 4], 3], [[6, 6, 6, 6], 4]]
            random_subnet: [[[4, 3, 2, 3], 1], [[4, 4, 4, 4], 1], [[6, 6, 6, 6], 1]]

        returns edit distance of architecture between the two subnets

        Weigh depth about twice as much.
        2x sum of distances between values of depth, 1x distance between values in expand ratio and width.
        Divide by 4 for expand ratio (because its 4 times as many values)
    '''
    elastic_ratio_multiplier = 1
    depth_multiplier = 2
    elastic_ratios_target = target_subnet[0]
    elastic_ratios_random = random_subnet[0]
    depths_target = target_subnet[1]
    depths_random = random_subnet[1]
    elastic_dist = 0
    for i, val in enumerate(elastic_ratios_target):
        dif = abs(val - elastic_ratios_random[i])
        elastic_dist += dif * elastic_ratio_multiplier

    depth_dist = 0
    for i, val in enumerate(depths_target):
        dif = abs(val - depths_random[i])
        depth_dist += dif * depth_multiplier

    edit_distance = elastic_dist + depth_dist
    return edit_distance

In [5]:
def get_shared_weights(net, smaller_subnet=(None, None, 4, 3), larger_subnet=(None, None, 6, 4)):
    '''
        This function will return a list of shared weights between two given subnetworks. 
        Each element (block element) of the returned list represents a block in the network with shared weights.
        Each block element is a list of all the tensors that are the shared weights
    '''
    if isinstance(net, nn.DataParallel):
        net = net.module

    net.set_active_subnet(*larger_subnet)
    larger_subnetwork = copy.deepcopy(net)
    #print(f"Num Parameters Largest {sum(p.numel() for p in net.parameters())}!\n")
    net.set_active_subnet(*smaller_subnet)
    #print(f"Num Parameters smallest {sum(p.numel() for p in net.parameters())}!\n")
    weights = []

    # While traversing the blocks, we need to keep track of input channels to get the proper active blocks
    # We need seperate ones because each sized network will have different out_channels for every block
    smaller_input_channel = net.blocks[0].mobile_inverted_conv.out_channels
    larger_input_channel = net.blocks[0].mobile_inverted_conv.out_channels
    count = 0
    for stage_id, block_idx in enumerate(net.block_group_info): # traverse through each block group and figure out how big each tensor is and how many tensors are in a group
        # block_idx is the index of the block in the stage
        depth = net.runtime_depth[stage_id] # number of active blocks in the stage
        active_idx = block_idx[:depth] # gets the indices of the active blocks based on runtime depth
        block_weights = []
        for idx in active_idx:
            # retrieves the active sub block for the smaller and large subnets
            smaller_block = net.blocks[idx].mobile_inverted_conv.get_active_subnet(smaller_input_channel, True)
            larger_block = larger_subnetwork.blocks[idx].mobile_inverted_conv.get_active_subnet(larger_input_channel, True)
            for larger_module, smaller_module in zip(larger_block.modules(), smaller_block.modules()):
                # We only care about the weights in the convolution layer
                if isinstance(larger_module, nn.Conv2d) and isinstance(smaller_module, nn.Conv2d):
                    for larger_param, smaller_param in zip(larger_module.parameters(), smaller_module.parameters()):
                        larger_shape = larger_param.shape
                        smaller_shape = smaller_param.shape
                        # try to find the overlap between the two tensors
                        # it calculates the overlapping region between the parameters of the larger and smaller subnet
                        overlap_size = tuple(min(smaller_shape[i], larger_shape[i]) for i in range(larger_param.dim()))

                        # Creates slices between the parameters of the larger and smaller subnets
                        # The size is determined by taking the minimum size along each dimension
                        slices = tuple(slice(0, s) for s in overlap_size)
                        overlapping_region = larger_param[slices]
                        overlapping_region_smaller = smaller_param[slices]

                        # Checks if the overlapping regions are identical, then the number of shared weights
                        # increases by the amount
                        if torch.equal(overlapping_region, overlapping_region_smaller):
                            count += overlapping_region_smaller.numel()
                            # block_weights.append(overlapping_region_smaller)
            smaller_input_channel = smaller_block.out_channels
            larger_input_channel = larger_block.out_channels
            # Updates the input channels for the next block based on the output channels of the current block
    return count

In [21]:
for key, value in od.items():
    print(f"Key: {key}, Values: {value}")

Key: 123.101904, Values: [((2, 2, 2, 2, 2), [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])]
Key: 138.303664, Values: [((2, 2, 3, 2, 2), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3])]
Key: 140.435424, Values: [((2, 2, 2, 2, 3), [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4])]
Key: 142.614352, Values: [((2, 3, 2, 2, 2), [3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])]
Key: 149.419216, Values: [((3, 2, 2, 2, 2), [4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])]
Key: 151.989712, Values: [((2, 2, 2, 3, 2), [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3])]
Key: 155.637184, Values: [((2, 2, 3, 2, 3), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 4, 4, 4, 4])]
Key: 157.816112, Values: [((2, 3, 3, 2, 2), [3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3])]
Key: 159.947872, Values: [((2, 3, 2, 2, 3), [3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4])]
Key: 164.620976, Values: [((3, 2, 3, 

In [22]:
interesting_keys = [166.752736, 167.191472, 168.707184, 168.931664, 169.323232, 171.50216]
interesting_keys2 = [181.674576, 181.954496, 184.133424, 184.524992, 186.040704, 186.265184]

In [23]:
for i in range(0, len(interesting_keys), 2):
    val_1 = od[interesting_keys[i]][0]
    val_2 = od[interesting_keys[i+1]][0]
    depth_1 = val_1[0]
    expand_ratio_1 = val_1[1]
    depth_2 = val_2[0]
    expand_ratio_2 = val_2[1]
    edit_distance = get_arch_edit_distance([expand_ratio_1, depth_1], [expand_ratio_2, depth_2])
    spd = get_shared_weights(mobilenet, [None, None, expand_ratio_2, depth_2], [None, None, expand_ratio_1, depth_1])
    print(f"Comparing FLOPS: {interesting_keys[i]} and {interesting_keys[i+1]}")
    print(f"Edit distance between\n\t{val_1} and\n\t{val_2} is\n\t{edit_distance}")
    print(f"Shared weights between\n\t{val_1} and\n\t{val_2} is\n\t{spd}")


Comparing FLOPS: 166.752736 and 167.191472
Edit distance between
	((3, 2, 2, 2, 3), [4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4]) and
	((2, 2, 3, 3, 2), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3]) is
	24
Shared weights between
	((3, 2, 2, 2, 3), [4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4]) and
	((2, 2, 3, 3, 2), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3]) is
	929160
Comparing FLOPS: 168.707184 and 168.931664
Edit distance between
	((2, 2, 4, 2, 2), [3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((3, 3, 2, 2, 2), [4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]) is
	28
Shared weights between
	((2, 2, 4, 2, 2), [3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((3, 3, 2, 2, 2), [4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]) is
	775288
Comparing FLOPS: 169.323232 and 171.50216
Edit distance between
	((2, 2, 2, 3, 3), [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4

In [25]:
for i in range(0, len(interesting_keys2), 2):
    val_1 = od[interesting_keys2[i]][0]
    val_2 = od[interesting_keys2[i+1]][0]
    depth_1 = val_1[0]
    expand_ratio_1 = val_1[1]
    depth_2 = val_2[0]
    expand_ratio_2 = val_2[1]
    edit_distance = get_arch_edit_distance([expand_ratio_1, depth_1], [expand_ratio_2, depth_2])
    spd = get_shared_weights(mobilenet, [None, None, expand_ratio_2, depth_2], [None, None, expand_ratio_1, depth_1])
    print(f"Comparing FLOPS: {interesting_keys2[i]} and {interesting_keys2[i+1]}")
    print(f"Edit distance between\n\t{val_1} and\n\t{val_2} is\n\t{edit_distance}")
    print(f"Shared weights between\n\t{val_1} and\n\t{val_2} is\n\t{spd}")


Comparing FLOPS: 181.674576 and 181.954496
Edit distance between
	((2, 4, 2, 2, 2), [3, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((3, 2, 3, 2, 3), [4, 4, 4, 4, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 4, 4, 4, 4]) is
	34
Shared weights between
	((2, 4, 2, 2, 2), [3, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((3, 2, 3, 2, 3), [4, 4, 4, 4, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 4, 4, 4, 4]) is
	1076816
Comparing FLOPS: 184.133424 and 184.524992
Edit distance between
	((3, 3, 3, 2, 2), [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((2, 2, 3, 3, 3), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]) is
	24
Shared weights between
	((3, 3, 3, 2, 2), [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3]) and
	((2, 2, 3, 3, 3), [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]) is
	1242760
Comparing FLOPS: 186.040704 and 186.265184
Edit distance between
	((2, 2, 4, 2, 3), [3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6, 3

In [6]:
min_net = (None , None, 3, 2)
max_net = (None , None, 6, 4)

mobilenet.set_active_subnet(*min_net)
sub = mobilenet.get_active_subnet(preserve_weight=True)
subnet_info = get_net_info(sub, measure_latency="gpu16")
flops_min = subnet_info['flops'] / 1e6

mobilenet.set_active_subnet(*max_net)
sub = mobilenet.get_active_subnet(preserve_weight=True)
subnet_info = get_net_info(sub, measure_latency="gpu16")
flops_max = subnet_info['flops'] / 1e6

edit_distance = get_arch_edit_distance([[3]*20, [2, 2, 2, 2, 2]], [[6]*20, [4, 4, 4, 4, 4]])
spd = get_shared_weights(mobilenet, min_net, max_net)

print(f"Max Net FLOPS: {flops_max} and Min Net FLOPS: {flops_min}")
print(f"Edit distance between\n\t{min_net} and\n\t{max_net} is\n\t{edit_distance}")
print(f"Shared weights between\n\t{min_net} and\n\t{max_net} is\n\t{spd}")

Total training params: 2.20M
Total FLOPs: 123.10M
Estimated gpu16 latency: 56.512ms
Total training params: 6.14M
Total FLOPs: 445.54M
Estimated gpu16 latency: 196.805ms
Max Net FLOPS: 445.53936 and Min Net FLOPS: 123.101904
Edit distance between
	(None, None, 3, 2) and
	(None, None, 6, 4) is
	80
Shared weights between
	(None, None, 3, 2) and
	(None, None, 6, 4) is
	750752
